# LeetCode 1024 — Video Stitching bằng Python

Link đề: https://leetcode.com/problems/video-stitching/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Greedy

Skeleton LeetCode:

```python
class Solution:
    def videoStitching(self, clips: List[List[int]], time: int) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất** (số clip tối thiểu, hoặc `-1`). Các clip là **đoạn đóng** có thể chồng lên nhau, được cắt tuỳ ý; hai clip chạm đúng một điểm (`[0,1]` và `[1,2]`) vẫn **nối liền** được. Điểm cốt lõi: bài toán **phủ đoạn `[0, time]` bằng ít đoạn nhất** — tham lam "nhảy xa nhất" (giống Jump Game II). Oracle độc lập: **tập điểm trên lưới nửa số nguyên** + duyệt mọi tập con clip.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chọn ít clip nhất sao cho hợp của chúng phủ kín `[0, time]`; không thể thì `-1`.
2. Nắm công cụ: mảng "với mỗi vị trí, đoạn kết thúc xa nhất của clip bắt đầu tại đó", tham lam nhảy xa nhất theo từng "tầng", quy hoạch động, BFS trên vị trí.
3. Đi từ **thử mọi tập con clip** → **tham lam bằng mảng `farthest` (kiểu Jump Game II)** → **sắp xếp theo điểm đầu + tham lam** → **quy hoạch động theo vị trí** → **BFS theo tầng**.
4. Hiểu mấu chốt: tại vị trí đã phủ tới `reach`, clip tiếp theo nên là clip có `start ≤ reach` và `end` **lớn nhất**; sau mỗi clip ta "nhảy" tới `end` đó.
5. Tránh lỗi kinh điển: dùng `<` thay `<=` khi nối clip (bỏ qua trường hợp chạm đầu mút), không kiểm tra có phủ được (`-1`), chọn clip dài nhất bất kể điểm đầu, bỏ sót yêu cầu bắt đầu từ `0`, đếm lệch một clip, sắp xếp theo điểm cuối thay vì điểm đầu.

## 2. Đề bài & ví dụ

Có chuỗi các clip video của một sự kiện thể thao dài `time` giây. Các clip có thể chồng lên nhau và độ dài khác nhau. `clips[i] = [start_i, end_i]`.

Ta được **cắt** clip tuỳ ý (ví dụ `[0, 7]` cắt thành `[0, 1] + [1, 3] + [3, 7]`). Trả về **số clip tối thiểu** để có thể cắt thành các đoạn phủ kín sự kiện `[0, time]`; nếu không thể, trả về `-1`.

**Ví dụ 1:** `clips = [[0,2],[4,6],[8,10],[1,9],[1,5],[5,9]]`, `time = 10` → `3` (lấy `[0,2]`, `[1,9]`, `[8,10]`).

**Ví dụ 2:** `clips = [[0,1],[1,2]]`, `time = 5` → `-1`.

**Ví dụ 3:** `clips = [[0,1],[6,8],[0,2],[5,6],[0,4],[0,3],[6,7],[1,3],[4,7],[1,4],[2,5],[2,6],[3,4],[4,5],[5,7],[6,9]]`, `time = 9` → `3`.

**Ràng buộc:** `1 <= clips.length <= 100`; `0 <= start_i <= end_i <= 100`; `1 <= time <= 100`.

In [1]:
import itertools
import random
import time as _time
from collections import deque
from typing import List

ex3 = [[0, 1], [6, 8], [0, 2], [5, 6], [0, 4], [0, 3], [6, 7], [1, 3], [4, 7], [1, 4], [2, 5], [2, 6], [3, 4], [4, 5], [5, 7], [6, 9]]
examples = [(([[0, 2], [4, 6], [8, 10], [1, 9], [1, 5], [5, 9]], 10), 3), (([[0, 1], [1, 2]], 5), -1), ((ex3, 9), 3), (([[0, 4]], 4), 1), (([[0, 1], [1, 2]], 2), 2)]
for (c, t), want in examples:
    print(f"time = {t}, clips = {c[:6]}{'...' if len(c) > 6 else ''} -> {want}")

time = 10, clips = [[0, 2], [4, 6], [8, 10], [1, 9], [1, 5], [5, 9]] -> 3
time = 5, clips = [[0, 1], [1, 2]] -> -1
time = 9, clips = [[0, 1], [6, 8], [0, 2], [5, 6], [0, 4], [0, 3]]... -> 3
time = 4, clips = [[0, 4]] -> 1
time = 2, clips = [[0, 1], [1, 2]] -> 2


## 3. Kiến thức nền

### 3.1 Phủ đoạn bằng đoạn đóng

Một tập đoạn đóng phủ `[0, T]` khi với mọi điểm thực `x ∈ [0, T]` có đoạn chứa `x`. Hai đoạn `[a, b]` và `[b, c]` **chạm** nhau tại `b` nên hợp của chúng là `[a, c]` liền mạch. Hai đoạn `[a, b]` và `[b + 1, c]` có khe `(b, b + 1)` — không liền.

### 3.2 Oracle bằng tập điểm trên lưới nửa số nguyên

Vì mọi đầu mút nguyên, ta biểu diễn `[a, b]` bằng tập số nguyên `{2a, 2a+1, …, 2b}` (nhân đôi toạ độ: bước `0.5`). Tập clip phủ `[0, T]` ⇔ hợp các tập điểm chứa `{0, …, 2T}`. Bước `0.5` đảm bảo khe `(b, b+1)` bị lộ ra (điểm `2b + 1` không thuộc tập nào).

In [2]:
def lattice_points(clip):
    return set(range(2 * clip[0], 2 * clip[1] + 1))


def covers(chosen, T):
    pts = set()
    for c in chosen:
        pts |= lattice_points(c)
    return all(p in pts for p in range(0, 2 * T + 1))


print(covers([[0, 1], [1, 2]], 2), covers([[0, 1], [2, 3]], 3), covers([[0, 2], [1, 9], [8, 10]], 10))

True False True


### 3.3 Điều kiện nối

Giả sử ta đã phủ được `[0, reach]`. Một clip `[s, e]` **mở rộng được** nếu `s ≤ reach` (đoạn đóng: `s = reach` vẫn nối) và `e > reach`; khi đó vùng phủ trở thành `[0, e]`.

## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: `time = 10`. Từ `reach = 0`: các clip bắt đầu tại `0` — chỉ có `[0,2]` ⇒ `reach = 2` (1 clip). Từ `reach = 2`: các clip có `start ≤ 2`: `[0,2]`, `[1,9]`, `[1,5]` ⇒ xa nhất `9` (2 clip). Từ `reach = 9`: `[8,10]`, `[5,9]`, `[1,9]` ⇒ `10` (3 clip) ≥ `time`. Đáp án `3`.

Ví dụ 2: sau `[0,1]`, `[1,2]` ta tới `2 < 5`, không còn clip nào tiếp ⇒ `-1`.

Mỗi lần chọn clip, **luôn chọn clip cho `reach` mới lớn nhất** — đó chính là tham lam kiểu Jump Game II.

In [3]:
def trace_greedy_steps(clips, T):
    reach, count = 0, 0
    print(f"time = {T}")
    while reach < T:
        cands = [c for c in clips if c[0] <= reach]
        best = max((c[1] for c in cands), default=-1)
        print(f"  reach = {reach}: các clip có start <= {reach} cho end xa nhất = {best}")
        if best <= reach:
            print("  không mở rộng được => -1")
            return -1
        reach, count = best, count + 1
    print("=>", count)
    return count


assert trace_greedy_steps(examples[0][0][0], 10) == 3
assert trace_greedy_steps([[0, 1], [1, 2]], 5) == -1

time = 10
  reach = 0: các clip có start <= 0 cho end xa nhất = 2
  reach = 2: các clip có start <= 2 cho end xa nhất = 9
  reach = 9: các clip có start <= 9 cho end xa nhất = 10
=> 3
time = 5
  reach = 0: các clip có start <= 0 cho end xa nhất = 1
  reach = 1: các clip có start <= 1 cho end xa nhất = 2
  reach = 2: các clip có start <= 2 cho end xa nhất = 2
  không mở rộng được => -1


## 5. Brute force — thử mọi tập con clip

**Ý tưởng:** theo đúng định nghĩa: duyệt các tập con theo kích thước tăng dần; tập đầu tiên phủ `[0, T]` (kiểm bằng `covers` trên lưới) cho đáp án. `O(2^n)` — chỉ cho `n ≲ 14`, dùng làm **oracle**.

In [4]:
def stitch_brute(clips, T):
    n = len(clips)
    for k in range(1, n + 1):
        for idx in itertools.combinations(range(n), k):
            if covers([clips[i] for i in idx], T):
                return k
    return -1


for (c, t), want in examples:
    if len(c) <= 16:
        assert stitch_brute(c, t) == want, (c, t)
print([stitch_brute(c, t) for (c, t), _ in examples[:2]], "(ví dụ 3 có 16 clip: bỏ qua ở đây, kiểm ở mục 12)")

[3, -1] (ví dụ 3 có 16 clip: bỏ qua ở đây, kiểm ở mục 12)


In [5]:
# Code nộp LeetCode cho Brute force (chỉ minh hoạ; 2^n tập con)
import itertools


class SolutionBrute:
    def videoStitching(self, clips: List[List[int]], time: int) -> int:
        n = len(clips)
        for k in range(1, n + 1):
            for idx in itertools.combinations(range(n), k):
                pts = set()
                for i in idx:
                    pts |= set(range(2 * clips[i][0], 2 * clips[i][1] + 1))
                if all(p in pts for p in range(2 * time + 1)):
                    return k
        return -1

**Độ phức tạp:** `O(2^n · n · time)`.

**Nút thắt:** ta thử cả tập con trong khi quyết định ở mỗi bước chỉ phụ thuộc vào `reach` hiện tại.

## 6. Bước đột phá

Với mỗi vị trí `i`, tính `farthest[i]` = điểm cuối lớn nhất của các clip có `start = i`. Duyệt `i` từ `0` tới `time − 1`, duy trì `reach` (xa nhất với được bằng số clip hiện tại) và `cur_end` (giới hạn của số clip đã chốt): khi `i` vượt `cur_end`, cần thêm một clip (`count += 1`, `cur_end = reach`); nếu `i > reach` thì `-1`. `O(n + time)`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** tham lam mảng `farthest` (đáp án chính), **Cách 3** sắp xếp theo điểm đầu + tham lam, **Cách 4** quy hoạch động theo vị trí, **Cách 5** BFS theo tầng.

### Cách 2 — Tham lam bằng mảng `farthest` (Jump Game II)

**Giải thích từng đoạn:**

1. `farthest = [0] * (time + 1)`; với mỗi clip `[s, e]` có `s ≤ time`: `farthest[s] = max(farthest[s], e)`.
2. `count = 0`, `cur_end = 0` (vùng chắc chắn phủ với `count` clip), `reach = 0` (xa nhất với được).
3. Với `i = 0..time − 1`: `reach = max(reach, farthest[i])` nếu `i ≤ reach` (nếu `i > reach` ⇒ `-1`); nếu `i == cur_end`: `count += 1`, `cur_end = reach` (chốt thêm một clip).
4. Cuối: `reach ≥ time` ⇒ `count`, ngược lại `-1`.

In [6]:
def stitch_farthest(clips, T):
    farthest = [0] * (T + 1)
    for s, e in clips:
        if s <= T:
            farthest[s] = max(farthest[s], e)
    count = cur_end = reach = 0
    for i in range(T):
        if i > reach:                                 # i không với tới: có khe hở
            return -1
        reach = max(reach, farthest[i])
        if i == cur_end:                              # hết vùng của các clip đã chốt: cần chốt thêm clip
            if reach <= i:                            # không mở rộng được
                return -1
            count += 1
            cur_end = reach
    return count if reach >= T else -1


for (c, t), want in examples:
    assert stitch_farthest(c, t) == want, (c, t, stitch_farthest(c, t), want)
print([stitch_farthest(c, t) for (c, t), _ in examples])

[3, -1, 3, 1, 2]


In [7]:
# Code nộp LeetCode cho Cách 2
class SolutionFarthest:
    def videoStitching(self, clips: List[List[int]], time: int) -> int:
        farthest = [0] * (time + 1)
        for s, e in clips:
            if s <= time:
                farthest[s] = max(farthest[s], e)
        count = cur_end = reach = 0
        for i in range(time):
            if i > reach:
                return -1
            reach = max(reach, farthest[i])
            if i == cur_end:
                if reach <= i:
                    return -1
                count += 1
                cur_end = reach
        return count if reach >= time else -1

**Độ phức tạp:** `O(n + time)` thời gian, `O(time)` bộ nhớ.

### Cách 3 — Sắp xếp theo điểm đầu + tham lam

**Ý tưởng:** sắp xếp clip theo `start`. Giữ `reach`. Lặp: trong các clip có `start ≤ reach` (con trỏ `i` chạy tới), lấy `end` lớn nhất `best`; nếu `best ≤ reach` ⇒ `-1`; ngược lại `reach = best`, `count += 1`; dừng khi `reach ≥ time`.

In [8]:
def stitch_sorted(clips, T):
    cl = sorted(clips)
    i, n = 0, len(cl)
    reach = count = 0
    while reach < T:
        best = reach
        while i < n and cl[i][0] <= reach:            # <= vì đoạn đóng chạm đầu mút vẫn nối
            best = max(best, cl[i][1])
            i += 1
        if best == reach:                             # không clip nào mở rộng được
            return -1
        reach = best
        count += 1
    return count


for (c, t), want in examples:
    assert stitch_sorted(c, t) == want
print([stitch_sorted(c, t) for (c, t), _ in examples])

[3, -1, 3, 1, 2]


In [9]:
# Code nộp LeetCode cho Cách 3
class SolutionSorted:
    def videoStitching(self, clips: List[List[int]], time: int) -> int:
        clips = sorted(clips)
        i, n = 0, len(clips)
        reach = count = 0
        while reach < time:
            best = reach
            while i < n and clips[i][0] <= reach:
                best = max(best, clips[i][1])
                i += 1
            if best == reach:
                return -1
            reach = best
            count += 1
        return count

**Độ phức tạp:** `O(n log n)` (sắp xếp) thời gian, `O(1)` bộ nhớ phụ.

### Cách 4 — Quy hoạch động theo vị trí

**Ý tưởng:** `dp[t]` = số clip tối thiểu để phủ `[0, t]` (`dp[0] = 0`, còn lại `∞`). Với `t = 1..time`: với mọi clip `[s, e]` có `s < t ≤ e`: `dp[t] = min(dp[t], dp[s] + 1)`. Đáp án `dp[time]` (hoặc `-1` nếu `∞`).

In [10]:
def stitch_dp(clips, T):
    INF = float("inf")
    dp = [INF] * (T + 1)
    dp[0] = 0
    for t in range(1, T + 1):
        for s, e in clips:
            if s < t <= e and dp[s] + 1 < dp[t]:
                dp[t] = dp[s] + 1
    return dp[T] if dp[T] != INF else -1


for (c, t), want in examples:
    assert stitch_dp(c, t) == want
print([stitch_dp(c, t) for (c, t), _ in examples])

[3, -1, 3, 1, 2]


In [11]:
# Code nộp LeetCode cho Cách 4
class SolutionDP:
    def videoStitching(self, clips: List[List[int]], time: int) -> int:
        INF = float("inf")
        dp = [INF] * (time + 1)
        dp[0] = 0
        for t in range(1, time + 1):
            for s, e in clips:
                if s < t <= e and dp[s] + 1 < dp[t]:
                    dp[t] = dp[s] + 1
        return dp[time] if dp[time] != INF else -1

**Độ phức tạp:** `O(time · n)` thời gian, `O(time)` bộ nhớ.

### Cách 5 — BFS theo tầng trên vị trí

**Ý tưởng:** trạng thái = `reach` (đã phủ `[0, reach]`). Từ `reach`, mỗi clip `[s, e]` với `s ≤ reach < e` dẫn tới trạng thái `e`. BFS từ `0`; số tầng tối thiểu để tới `reach ≥ time` là đáp án.

In [12]:
def stitch_bfs(clips, T):
    dist = {0: 0}
    queue = deque([0])
    while queue:
        r = queue.popleft()
        if r >= T:
            return dist[r]
        for s, e in clips:
            if s <= r < e and e not in dist:
                dist[e] = dist[r] + 1
                queue.append(e)
    return -1


for (c, t), want in examples:
    assert stitch_bfs(c, t) == want
print([stitch_bfs(c, t) for (c, t), _ in examples])

[3, -1, 3, 1, 2]


In [13]:
# Code nộp LeetCode cho Cách 5
from collections import deque


class SolutionBFS:
    def videoStitching(self, clips: List[List[int]], time: int) -> int:
        dist = {0: 0}
        queue = deque([0])
        while queue:
            r = queue.popleft()
            if r >= time:
                return dist[r]
            for s, e in clips:
                if s <= r < e and e not in dist:
                    dist[e] = dist[r] + 1
                    queue.append(e)
        return -1

**Độ phức tạp:** `O(V · n)` với `V ≤ 101` trạng thái (`reach ∈ [0, 100]`) ⇒ `O(n · time)`.

## 8. TRACE từng bước

**Trace Cách 2** với ví dụ 1 (`time = 10`): `farthest[0] = 2`, `farthest[1] = 9` (từ `[1,9]`, `[1,5]`), `farthest[4] = 6`, `farthest[5] = 9`, `farthest[8] = 10`.

| `i` | `reach` trước | `reach = max(reach, farthest[i])` | `i == cur_end`? | `count` | `cur_end` |
|---|---|---|---|---|---|
| 0 | 0 | 2 | có (0 = 0) | 1 | 2 |
| 1 | 2 | 9 | không (1 ≠ 2) | 1 | 2 |
| 2 | 9 | 9 | có | 2 | 9 |
| 3..7 | 9 | 9 | không | 2 | 9 |
| 8 | 9 | 10 | không | 2 | 9 |
| 9 | 10 | 10 | có (9 = 9) | 3 | 10 |

`reach = 10 ≥ time` ⇒ đáp án `3` ✓. Hàm trace in từng bước:

In [14]:
def trace_farthest(clips, T):
    farthest = [0] * (T + 1)
    for s, e in clips:
        if s <= T:
            farthest[s] = max(farthest[s], e)
    print("farthest =", farthest)
    count = cur_end = reach = 0
    for i in range(T):
        if i > reach:
            print(f"  i={i} > reach={reach}: khe hở => -1")
            return -1
        reach = max(reach, farthest[i])
        note = ""
        if i == cur_end:
            if reach <= i:
                print(f"  i={i}: không mở rộng được => -1")
                return -1
            count += 1
            cur_end = reach
            note = f" -> chốt clip thứ {count}, cur_end = {cur_end}"
        print(f"  i={i}: reach={reach}{note}")
    res = count if reach >= T else -1
    print("=>", res)
    return res


assert trace_farthest(examples[0][0][0], 10) == 3

farthest = [2, 9, 0, 0, 6, 9, 0, 0, 10, 0, 0]
  i=0: reach=2 -> chốt clip thứ 1, cur_end = 2
  i=1: reach=9
  i=2: reach=9 -> chốt clip thứ 2, cur_end = 9
  i=3: reach=9
  i=4: reach=9
  i=5: reach=9
  i=6: reach=9
  i=7: reach=9
  i=8: reach=10
  i=9: reach=10 -> chốt clip thứ 3, cur_end = 10
=> 3


**Trace chạm đầu mút:** `clips = [[0,1],[1,2]]`, `time = 2`: `farthest = [1, 2, 0]`. `i = 0`: `reach = 1`, chốt clip 1 (`cur_end = 1`); `i = 1`: `reach = 2`, `i == cur_end` ⇒ chốt clip 2 ⇒ `2`. Điều kiện `i > reach` (không phải `i ≥ reach`) cho phép `[0,1]` và `[1,2]` nối.

In [15]:
assert stitch_farthest([[0, 1], [1, 2]], 2) == 2 and stitch_sorted([[0, 1], [1, 2]], 2) == 2 and stitch_dp([[0, 1], [1, 2]], 2) == 2
print("chạm đầu mút vẫn nối:", stitch_farthest([[0, 1], [1, 2]], 2), "| có khe (2 -> 3):", stitch_farthest([[0, 1], [2, 3]], 3))

chạm đầu mút vẫn nối: 2 | có khe (2 -> 3): -1


## 9. Code nộp LeetCode

Chọn **Cách 2** (tham lam mảng `farthest`). Giữ nguyên `class Solution`, `videoStitching(self, clips, time)`:

In [16]:
class Solution:
    def videoStitching(self, clips: List[List[int]], time: int) -> int:
        farthest = [0] * (time + 1)                    # farthest[i]: end xa nhất của clip bắt đầu tại i
        for s, e in clips:
            if s <= time:
                farthest[s] = max(farthest[s], e)
        count = cur_end = reach = 0
        for i in range(time):
            if i > reach:                              # khe hở: không clip nào phủ tới i
                return -1
            reach = max(reach, farthest[i])
            if i == cur_end:                           # hết vùng của các clip đã chốt: chốt thêm clip xa nhất
                if reach <= i:
                    return -1
                count += 1
                cur_end = reach
        return count if reach >= time else -1


print([Solution().videoStitching([c[:] for c in cl], t) for (cl, t), _ in examples])
assert all(Solution().videoStitching([c[:] for c in cl], t) == w for (cl, t), w in examples)

[3, -1, 3, 1, 2]


## 10. Vì sao đúng

**(a) Bài toán phủ.** Tập clip phủ `[0, T]` ⇔ sắp theo điểm đầu có dãy chain: clip đầu bắt đầu tại `0`, mỗi clip kế bắt đầu `≤` điểm cuối lớn nhất đã đạt, và điểm cuối lớn nhất cuối cùng `≥ T`.

**(b) Tham lam đúng (trao đổi).** Gọi `r_k` là `reach` lớn nhất đạt được bằng đúng `k` clip theo tham lam (mỗi bước lấy clip có `start ≤ r_{k−1}` và `end` lớn nhất). Quy nạp: **mọi** lời giải dùng `k` clip có `reach ≤ r_k`. Cơ sở `k = 0`: `r_0 = 0`. Bước: lời giải tối ưu dùng `k` clip có clip thứ `k` bắt đầu tại `s ≤ (reach sau k − 1 clip) ≤ r_{k−1}` nên `end ≤ max{e : s ≤ r_{k−1}} = r_k`. Vậy `k` nhỏ nhất với `r_k ≥ T` là tối ưu; nếu `r_k = r_{k−1}` (không mở rộng) thì mọi `k' ≥ k` cũng không ⇒ `-1`.

**(c) Cách 2 mô phỏng (b).** `cur_end` là `r_count`; `reach` là `max(farthest[j] : j ≤ i)` = clip tốt nhất với `start ≤ i`. Khi `i` chạm `cur_end`, ta dùng thêm một clip để nhảy tới `reach = r_{count+1}`. Điều kiện `i > reach` phát hiện khe hở. `farthest[s] = max(end)` đủ vì với cùng `start` chỉ clip dài nhất có ích.

**(d) Cách 3.** Cùng ý nhưng gom clip theo `start ≤ reach` từ danh sách đã sắp (mỗi clip duyệt một lần).

**(e) Cách 4.** `dp[t]` = số clip tối thiểu phủ `[0, t]`. Clip cuối cùng `[s, e]` của một lời giải tối ưu phủ `[0, t]` có `s < t ≤ e` (để chứa đoạn cuối `(…, t]`), và phần còn lại phải phủ `[0, s]` nên tốn ít nhất `dp[s]` clip; ngược lại `dp[s] + 1` đạt được. Do đó công thức đúng. (Chú ý `dp` không giảm theo `t` nên phủ `[0, s]` là đủ.)

**(f) Cách 5.** BFS trên đồ thị trạng thái `reach` với cạnh `reach → e` khi có clip `[s, e]`, `s ≤ reach < e`: số cạnh tối thiểu tới `reach ≥ T`.

**(g) Kiểm chứng bằng máy.** Mọi cách khớp oracle tập con + tập điểm trên **mọi** tập clip chọn từ `15` clip có đầu mút `0..4` và mọi `time ∈ [1, 4]` (`131 072` ca) và trên ví dụ ngẫu nhiên tới `n = 100`, `time = 100` (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mọi tập con | `O(2^n · n · T)` | `O(T)` | oracle `n ≲ 14` |
| 2. Tham lam `farthest` | `O(n + T)` | `O(T)` | **đáp án chính** |
| 3. Sắp xếp + tham lam | `O(n log n)` | `O(1)` | không mảng phụ |
| 4. DP theo vị trí | `O(n · T)` | `O(T)` | dễ chứng minh |
| 5. BFS theo tầng | `O(n · T)` | `O(T)` | cách nhìn đồ thị |

Đo thực tế (`n = 100`, `time = 100`):

In [17]:
def bench(fn, args, repeat=200):
    best = 1e9
    for _ in range(3):
        t0 = _time.perf_counter()
        for _ in range(repeat):
            fn(*args)
        best = min(best, (_time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(1024)
big = []
for _ in range(100):
    s = rng.randint(0, 99)
    big.append([s, min(100, s + rng.randint(0, 8))])
big.append([0, 3])
for name, fn in [("farthest", stitch_farthest), ("sorted", stitch_sorted), ("dp", stitch_dp), ("bfs", stitch_bfs)]:
    print(f"{name:9s}: {bench(fn, (big, 100)) * 1e6:8.1f} us -> {fn(big, 100)}")

farthest :     19.9 us -> 19
sorted   :     37.5 us -> 19


dp       :    407.5 us -> 19
bfs      :    196.6 us -> 19


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi tập clip từ 15 clip có đầu mút `0..4`, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "farthest": stitch_farthest,
    "sorted": stitch_sorted,
    "dp": stitch_dp,
    "bfs": stitch_bfs,
    "brute": stitch_brute,
    "Solution": lambda c, t: Solution().videoStitching([x[:] for x in c], t),
    "SolutionFarthest": lambda c, t: SolutionFarthest().videoStitching([x[:] for x in c], t),
    "SolutionSorted": lambda c, t: SolutionSorted().videoStitching([x[:] for x in c], t),
    "SolutionDP": lambda c, t: SolutionDP().videoStitching([x[:] for x in c], t),
    "SolutionBFS": lambda c, t: SolutionBFS().videoStitching([x[:] for x in c], t),
    "SolutionBrute": lambda c, t: SolutionBrute().videoStitching([x[:] for x in c], t),
}
SLOW = {"brute", "SolutionBrute"}
fast = [n for n in IMPLS if n not in SLOW]


def check(clips, T, want=None, slow_n=12):
    if want is None and len(clips) <= slow_n:
        want = stitch_brute(clips, T)
    results = {}
    snap = [c[:] for c in clips]
    for name in IMPLS:
        if name in SLOW and len(clips) > slow_n:
            continue
        results[name] = IMPLS[name](clips, T)
    assert clips == snap
    assert len(set(results.values())) == 1, (clips[:6], T, results)
    got = next(iter(results.values()))
    if want is not None:
        assert got == want, (clips[:6], T, got, want)
    return got


for (c, t), want in examples:
    check(c, t, want)
print("ví dụ đề: các cài đặt (brute chỉ cho ≤ 12 clip) đúng, không sửa đầu vào")
assert stitch_brute(ex3[:12], 9) in (-1, 3, 4, 5)

ví dụ đề: các cài đặt (brute chỉ cho ≤ 12 clip) đúng, không sửa đầu vào


### B. Edge case

- `time = 1`: một clip `[0, 1]`; hoặc `[0, 5]` (dài hơn time).
- Không có clip bắt đầu ở `0` ⇒ `-1`.
- Chạm đầu mút (`[0,1]`, `[1,2]`) vẫn nối; có khe (`[0,1]`, `[2,3]`) ⇒ `-1`.
- Clip dài hơn `time` vẫn dùng được; clip bắt đầu ở `time` (vô dụng).
- Clip có `start = end` (độ dài `0`): `[0,0]`, `[3,3]`.
- Trùng lặp clip.
- Cần clip "bắc cầu" (tham lam theo end lớn nhất): `[0,5], [1,2], [4,10]`.

In [19]:
edge = [
    ([[0, 1]], 1), ([[0, 5]], 1), ([[1, 3]], 3), ([[0, 1], [1, 2]], 2), ([[0, 1], [2, 3]], 3), ([[0, 0], [0, 3]], 3), ([[0, 3], [3, 3]], 3), ([[0, 2], [2, 2], [2, 4]], 4),
    ([[0, 5], [1, 2], [4, 10]], 10), ([[0, 2], [0, 2], [2, 4], [2, 4]], 4), ([[0, 1], [1, 3], [3, 6]], 6), ([[0, 4], [2, 6], [4, 8], [6, 10]], 10), ([[5, 9]], 9),
    ([[0, 100]], 100), ([[i, i + 1] for i in range(100)], 100), ([[i, i + 1] for i in range(1, 100)], 100), ([[0, 10], [10, 20]], 20), ([[0, 10], [11, 20]], 20),
]
for c, t in edge:
    check(c, t)
assert Solution().videoStitching([[i, i + 1] for i in range(100)], 100) == 100
assert Solution().videoStitching([[i, i + 1] for i in range(1, 100)], 100) == -1
print(len(edge), "edge case (kể cả 100 clip dài 1 và thiếu clip đầu): đúng")

18 edge case (kể cả 100 clip dài 1 và thiếu clip đầu): đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_strict_less(clips, T):                          # nối clip chỉ khi start < reach (bỏ chạm đầu mút)
    cl = sorted(clips)
    i, n = 0, len(cl)
    reach = count = 0
    while reach < T:
        best = reach
        while i < n and (cl[i][0] < reach or (cl[i][0] == 0 and reach == 0)):
            best = max(best, cl[i][1])
            i += 1
        if best == reach:
            return -1
        reach = best
        count += 1
    return count


def bug_longest_first(clips, T):                        # luôn lấy clip dài nhất bất kể điểm đầu
    cl = sorted(clips, key=lambda c: c[0] - c[1])
    reach = count = 0
    for s, e in cl:
        if reach >= T:
            break
        reach = max(reach, e)
        count += 1
    return count if reach >= T else -1


def bug_no_minus_one(clips, T):                         # không kiểm tra khe hở
    cl = sorted(clips)
    i, n = 0, len(cl)
    reach = count = 0
    while reach < T and i < n:
        best = reach
        while i < n and cl[i][0] <= reach:
            best = max(best, cl[i][1])
            i += 1
        if best == reach:
            break
        reach = best
        count += 1
    return count


def bug_sort_by_end(clips, T):                          # tham lam theo điểm cuối nhỏ nhất
    cl = sorted(clips, key=lambda c: c[1])
    reach = count = 0
    for s, e in cl:
        if s <= reach and e > reach:
            reach = e
            count += 1
        if reach >= T:
            return count
    return -1


def bug_count_off_by_one(clips, T):
    r = stitch_sorted(clips, T)
    return r + 1 if r > 0 else r


def bug_no_start_zero(clips, T):                        # không đòi clip đầu tiên bắt đầu tại 0 (bắt đầu từ clip sớm nhất)
    cl = sorted(clips)
    reach = cl[0][1]
    count = 1
    i = 1
    while reach < T:
        best = reach
        while i < len(cl) and cl[i][0] <= reach:
            best = max(best, cl[i][1])
            i += 1
        if best == reach:
            return -1
        reach, count = best, count + 1
    return count


def bug_dp_init_one(clips, T):                          # dp[0] = 1
    INF = float("inf")
    dp = [INF] * (T + 1)
    dp[0] = 1
    for t in range(1, T + 1):
        for s, e in clips:
            if s < t <= e and dp[s] + 1 < dp[t]:
                dp[t] = dp[s] + 1
    return dp[T] if dp[T] != INF else -1


BUGS = [bug_strict_less, bug_longest_first, bug_no_minus_one, bug_sort_by_end, bug_count_off_by_one, bug_no_start_zero, bug_dp_init_one]
probe = [([[0, 1], [1, 2]], 2), ([[0, 1], [1, 2]], 5), ([[1, 3]], 3), ([[0, 2], [4, 6], [8, 10], [1, 9], [1, 5], [5, 9]], 10), ([[0, 5], [1, 2], [4, 10]], 10),
         ([[0, 3], [2, 4], [3, 8], [1, 2]], 8), ([[0, 1], [2, 3]], 3), ([[0, 4], [2, 6], [4, 8]], 8), ([[1, 5], [0, 2]], 5)]
for bug in BUGS:
    found = None
    for c, t in probe:
        try:
            got = bug([x[:] for x in c], t)
        except Exception as e:
            got = type(e).__name__
        if got != stitch_brute(c, t):
            found = (c, t, got)
            break
    assert found, bug.__name__
    print(f"{bug.__name__:20s} sai trên clips={found[0]}, time={found[1]}: ra {found[2]}, đúng {stitch_brute(found[0], found[1])}")

bug_strict_less      sai trên clips=[[0, 1], [1, 2]], time=2: ra -1, đúng 2
bug_longest_first    sai trên clips=[[1, 3]], time=3: ra 1, đúng -1
bug_no_minus_one     sai trên clips=[[0, 1], [1, 2]], time=5: ra 2, đúng -1
bug_sort_by_end      sai trên clips=[[0, 2], [4, 6], [8, 10], [1, 9], [1, 5], [5, 9]], time=10: ra 5, đúng 3
bug_count_off_by_one sai trên clips=[[0, 1], [1, 2]], time=2: ra 3, đúng 2
bug_no_start_zero    sai trên clips=[[1, 3]], time=3: ra 1, đúng -1
bug_dp_init_one      sai trên clips=[[0, 1], [1, 2]], time=2: ra 3, đúng 2


### D. Vét cạn mọi tập clip từ 15 clip có đầu mút `0..4`

Có `15` clip `[a, b]` với `0 ≤ a ≤ b ≤ 4` (kể cả độ dài `0`). Mỗi **tập con** của 15 clip (`2^15 = 32 768` tập) là một đầu vào; với mỗi `time ∈ {1, 2, 3, 4}` oracle theo **định nghĩa**: `best[S]` = số clip nhỏ nhất của một tập con của `S` phủ `[0, time]` (phủ kiểm bằng tập điểm lưới nửa số nguyên), tính bằng quy hoạch trên mặt nạ bit. Tổng `4 × 32 767` ca.

In [21]:
UNIV = [[a, b] for a in range(5) for b in range(a, 5)]
assert len(UNIV) == 15
t0 = _time.perf_counter()
cnt = 0
main_impls = ["farthest", "sorted", "dp", "bfs", "Solution"]
INF = 99
for T in range(1, 5):
    full = (1 << (2 * T + 1)) - 1
    pts = [0] * 15
    for i, c in enumerate(UNIV):
        for p in range(2 * c[0], 2 * c[1] + 1):
            if p <= 2 * T:
                pts[i] |= 1 << p
    covered = [0] * (1 << 15)
    best = [INF] * (1 << 15)
    for mask in range(1, 1 << 15):
        low = mask & -mask
        i = low.bit_length() - 1
        covered[mask] = covered[mask ^ low] | pts[i]
        b = bin(mask).count("1") if covered[mask] == full else INF
        for j in range(15):
            if mask >> j & 1:
                b = min(b, best[mask ^ (1 << j)])
        best[mask] = b
    for mask in range(1, 1 << 15):
        clips = [UNIV[i] for i in range(15) if mask >> i & 1]
        want = best[mask] if best[mask] != INF else -1
        for name in main_impls:
            assert IMPLS[name](clips, T) == want, (name, clips, T, IMPLS[name](clips, T), want)
        cnt += 1
print(cnt, "ca (mọi tập con của 15 clip × time ∈ 1..4):", len(main_impls), "cài đặt khớp oracle định nghĩa;", round(_time.perf_counter() - t0, 1), "s")
assert cnt == 4 * (2 ** 15 - 1)

131068 ca (mọi tập con của 15 clip × time ∈ 1..4): 5 cài đặt khớp oracle định nghĩa; 1.6 s


In [22]:
# brute (tập con theo kích thước) và các class còn lại trên mẫu cố định của các tập con
rng = random.Random(102401)
others = ["SolutionFarthest", "SolutionSorted", "SolutionDP", "SolutionBFS", "SolutionBrute", "brute"]
for _ in range(2500):
    k = rng.randint(1, 9)
    clips = [UNIV[i][:] for i in rng.sample(range(15), k)]
    T = rng.randint(1, 4)
    want = stitch_farthest(clips, T)
    for name in others:
        assert IMPLS[name](clips, T) == want, (name, clips, T)
print("2500 ca mẫu: brute và các class còn lại khớp")

2500 ca mẫu: brute và các class còn lại khớp


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Các cách nhanh khớp nhau; với `n ≤ 12` khớp brute; (2) thêm một clip **không làm đáp án tăng** (và có thể biến `-1` thành số); (3) xáo trộn thứ tự clip không đổi đáp án; (4) trùng lặp clip không đổi đáp án; (5) đáp án `≤ time` (mỗi clip phủ ít nhất một đơn vị khi cần); (6) nếu đáp án `k ≠ -1` thì tồn tại `k` clip thật sự phủ `[0, time]` (truy vết từ tham lam rồi kiểm bằng `covers`); (7) tăng `time` không làm đáp án giảm (khi cả hai hữu hạn).

In [23]:
def pick_clips(clips, T):
    """Truy vết các clip được chọn bởi tham lam (dùng để kiểm chứng đáp án)."""
    cl = sorted(clips)
    i, n = 0, len(cl)
    reach, chosen = 0, []
    while reach < T:
        best, best_c = reach, None
        while i < n and cl[i][0] <= reach:
            if cl[i][1] > best:
                best, best_c = cl[i][1], cl[i]
            i += 1
        if best_c is None:
            return None
        chosen.append(best_c)
        reach = best
    return chosen


def random_clips(rng, n, T, maxlen):
    out = []
    for _ in range(n):
        s = rng.randint(0, T)
        out.append([s, min(T + rng.randint(0, 3), s + rng.randint(0, maxlen))])
    return out


rng = random.Random(20240114)
t0 = _time.perf_counter()
cnt_pos = cnt_neg = 0
for it in range(3000):
    T = rng.choice([1, 2, 5, 10, 30, 100])
    n = rng.choice([1, 3, 6, 12, 30, 100])
    clips = random_clips(rng, n, T, rng.choice([1, 3, 10, 50]))
    res = {name: IMPLS[name](clips, T) for name in fast}
    assert len(set(res.values())) == 1, (clips, T, res)                                  # (1)
    ans = res["Solution"]
    if n <= 12:
        assert ans == stitch_brute(clips, T)                                             # (1)
    sh = clips[:]
    rng.shuffle(sh)
    assert Solution().videoStitching(sh, T) == ans                                       # (3)
    assert Solution().videoStitching(clips + clips, T) == ans                            # (4)
    extra = random_clips(rng, 1, T, 5)
    a2 = Solution().videoStitching(clips + extra, T)
    assert (ans == -1) or (a2 != -1 and a2 <= ans)                                       # (2)
    if ans != -1:
        cnt_pos += 1
        assert ans <= T                                                                  # (5)
        chosen = pick_clips(clips, T)
        assert chosen is not None and len(chosen) == ans and covers(chosen, T)           # (6)
        if T < 100:
            a3 = Solution().videoStitching(clips, T + 1)
            assert a3 == -1 or a3 >= ans                                                 # (7)
    else:
        cnt_neg += 1
print("3000 ca ngẫu nhiên (n tới 100, time tới 100): 7 tính chất đúng; có đáp án:", cnt_pos, "| -1:", cnt_neg, ";", round(_time.perf_counter() - t0, 1), "s")

3000 ca ngẫu nhiên (n tới 100, time tới 100): 7 tính chất đúng; có đáp án: 1367 | -1: 1633 ; 3.8 s


### F. Giới hạn

`n = 100`, `time = 100`: 100 clip dài 1 (cần đúng 100), các clip dài xen kẽ cần ít clip, trường hợp khe hở ở cuối (`-1`), một clip `[0, 100]`.

In [24]:
t0 = _time.perf_counter()
cases = [
    ([[i, i + 1] for i in range(100)], 100, 100), ([[0, 100]], 100, 1), ([[i, i + 2] for i in range(0, 100, 2)], 100, 50), ([[i, i + 1] for i in range(99)], 100, -1),
    ([[0, 50], [50, 100]] + [[i, i + 1] for i in range(100)], 100, 2), ([[0, 49], [50, 100]], 100, -1), ([[0, 49], [49, 100]], 100, 2),
    ([[i, min(100, i + 11)] for i in range(0, 100, 10)], 100, 10), ([[0, 1]] + [[i, i + 3] for i in range(1, 98, 3)], 100, None),
]
for c, T, want in cases:
    res = {name: IMPLS[name](c, T) for name in fast}
    assert len(set(res.values())) == 1, (T, res)
    if want is not None:
        assert res["Solution"] == want, (c[:3], T, res["Solution"], want)
    print(f"  n = {len(c):3d}, time = {T}: đáp án = {res['Solution']}")
print("n ≤ 100, time ≤ 100: mọi cài đặt nhanh khớp nhau và khớp đáp án dựng sẵn;", round(_time.perf_counter() - t0, 2), "s")

  n = 100, time = 100: đáp án = 100
  n =   1, time = 100: đáp án = 1
  n =  50, time = 100: đáp án = 50
  n =  99, time = 100: đáp án = -1
  n = 102, time = 100: đáp án = 2
  n =   2, time = 100: đáp án = -1
  n =   2, time = 100: đáp án = 2
  n =  10, time = 100: đáp án = 10
  n =  34, time = 100: đáp án = 34
n ≤ 100, time ≤ 100: mọi cài đặt nhanh khớp nhau và khớp đáp án dựng sẵn; 0.01 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `<` thay `<=` khi nối clip

In [25]:
c, t = [[0, 1], [1, 2]], 2
print("bug:", bug_strict_less(c, t), "| đúng:", Solution().videoStitching(c, t))
assert bug_strict_less(c, t) == -1 and Solution().videoStitching(c, t) == 2

bug: -1 | đúng: 2


**Vì sao sai:** đoạn **đóng** `[0, 1]` và `[1, 2]` chạm nhau tại `1` nên nối liền. **Đúng:** `start ≤ reach`.

### Lỗi 2: chọn clip dài nhất bất kể điểm đầu

In [26]:
c, t = [[0, 5], [1, 2], [4, 10]], 10
print("bug:", bug_longest_first(c, t), "| đúng:", Solution().videoStitching(c, t))
c2, t2 = [[0, 3], [2, 4], [3, 8], [1, 2]], 8
print("bug:", bug_longest_first(c2, t2), "| đúng:", Solution().videoStitching(c2, t2))
assert bug_longest_first(c, t) == 1 and Solution().videoStitching(c, t) == 2       # [0,5] rồi [4,10]; bản sai chỉ đếm clip dài nhất [4,10] (không nối từ 0)
assert bug_longest_first(c2, t2) == 1 and Solution().videoStitching(c2, t2) == 2

bug: 1 | đúng: 2
bug: 1 | đúng: 2


**Vì sao sai:** clip dài nhất có thể bắt đầu xa chỗ ta đã phủ; chỉ được dùng clip có `start ≤ reach`.

### Lỗi 3: không trả `-1` khi có khe hở

In [27]:
c, t = [[0, 1], [2, 3]], 3
print("bug:", bug_no_minus_one(c, t), "| đúng:", Solution().videoStitching(c, t))
assert bug_no_minus_one(c, t) == 1 and Solution().videoStitching(c, t) == -1

bug: 1 | đúng: -1


**Vì sao sai:** khi không clip nào mở rộng được vùng phủ, mà `reach < time`, phải trả `-1`.

### Lỗi 4: tham lam theo điểm cuối

In [28]:
bad = next((c, t) for c, t in probe if bug_sort_by_end(c, t) != stitch_brute(c, t))
print("phản ví dụ:", bad, "| bug:", bug_sort_by_end(*bad), "| đúng:", stitch_brute(*bad))
assert bug_sort_by_end(*bad) != stitch_brute(*bad)

phản ví dụ: ([[0, 2], [4, 6], [8, 10], [1, 9], [1, 5], [5, 9]], 10) | bug: 5 | đúng: 3


**Vì sao sai:** đoạn kết thúc sớm nhất không phải lựa chọn tốt; ta muốn **mở rộng tối đa** vùng phủ (end lớn nhất) trong số các clip nối được.

### Lỗi 5: đếm lệch một clip / `dp[0] = 1`

In [29]:
c, t = [[0, 4]], 4
print("lệch 1:", bug_count_off_by_one(c, t), "| dp[0]=1:", bug_dp_init_one(c, t), "| đúng:", Solution().videoStitching(c, t))
assert bug_count_off_by_one(c, t) == 2 and bug_dp_init_one(c, t) == 2 and Solution().videoStitching(c, t) == 1

lệch 1: 2 | dp[0]=1: 2 | đúng: 1


**Vì sao sai:** vùng phủ rỗng `[0, 0]` tốn `0` clip (`dp[0] = 0`); mỗi clip chốt thêm đúng `1`.

### Lỗi 6 (bonus): bỏ qua yêu cầu bắt đầu từ `0`

In [30]:
c, t = [[1, 5]], 5
print("bug:", bug_no_start_zero(c, t), "| đúng:", Solution().videoStitching(c, t))
assert bug_no_start_zero(c, t) == 1 and Solution().videoStitching(c, t) == -1

bug: 1 | đúng: -1


**Vì sao sai:** phải phủ **từ giây `0`**; nếu không có clip bắt đầu ở `0` thì khoảng `[0, 1)` không được phủ.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "ít đoạn nhất phủ kín một khoảng" → tham lam "nhảy xa nhất" (Jump Game II) hoặc DP theo vị trí.

**Mẹo:**

1. `farthest[start] = max(end)`; duyệt vị trí tăng dần.
2. Nối được khi `start ≤ reach` (đoạn đóng).
3. `cur_end`/`reach`: khi `i == cur_end` thì chốt thêm clip.
4. Kiểm khe hở `i > reach` và `reach ≥ time` cuối cùng.
5. Oracle: tập điểm trên lưới nửa số nguyên + duyệt tập con.
6. Chuyển về Jump Game II: `farthest[i]` là "bước nhảy tối đa" từ `i`.

**Họ bài cùng pattern:** Jump Game II (45), Minimum Number of Taps to Open to Water a Garden (1326), Minimum Number of Arrows to Burst Balloons (452), Non-overlapping Intervals (435), Merge Intervals (56), Meeting Rooms II (253, Premium).

## 15. Biến thể

1. **Phủ đoạn `[a, b]` bất kỳ** (không nhất thiết bắt đầu từ `0`).
2. **Jump Game II** (45) quy về bài này.
3. **Trả về các clip được chọn** và kiểm chứng phủ.

Cài đặt và đối chiếu oracle:

In [31]:
def min_cover(clips, a, b):
    """Số đoạn tối thiểu phủ [a, b] (tham lam)."""
    cl = sorted(clips)
    i, n = 0, len(cl)
    reach, count = a, 0
    while reach < b:
        best = reach
        while i < n and cl[i][0] <= reach:
            best = max(best, cl[i][1])
            i += 1
        if best == reach:
            return -1
        reach, count = best, count + 1
    return count


def covers_ab(chosen, a, b):
    pts = set()
    for c in chosen:
        pts |= lattice_points(c)
    return all(p in pts for p in range(2 * a, 2 * b + 1))


def brute_cover(clips, a, b):
    n = len(clips)
    for k in range(0, n + 1):
        for idx in itertools.combinations(range(n), k):
            if covers_ab([clips[i] for i in idx], a, b):
                return k
    return -1


def jump_game_ii(nums):
    """Biến thể 2: số bước nhảy tối thiểu từ chỉ số 0 tới n-1 (luôn tới được); quy về video stitching."""
    n = len(nums)
    if n == 1:
        return 0
    clips = [[i, i + nums[i]] for i in range(n)]               # từ i nhảy được tới tối đa i + nums[i]
    return Solution().videoStitching(clips, n - 1)


def jump_brute(nums):
    n = len(nums)
    dist = {0: 0}
    queue = deque([0])
    while queue:
        i = queue.popleft()
        if i == n - 1:
            return dist[i]
        for j in range(i + 1, min(n - 1, i + nums[i]) + 1):
            if j not in dist:
                dist[j] = dist[i] + 1
                queue.append(j)
    return -1


rng = random.Random(15)
for _ in range(400):
    n = rng.randint(1, 8)
    a = rng.randint(0, 3)
    b = a + rng.randint(1, 4)
    clips = [[s, s + rng.randint(0, 3)] for s in (rng.randint(0, 6) for _ in range(n))]
    assert min_cover(clips, a, b) == brute_cover(clips, a, b)                          # (1)
nums_ok = 0
for _ in range(400):
    L = rng.randint(1, 9)
    nums = [rng.randint(1, 3) for _ in range(L)]
    assert jump_game_ii(nums) == jump_brute(nums)                                      # (2)
    nums_ok += 1
print("phủ [a, b] bất kỳ và Jump Game II quy về video stitching: khớp vét cạn;", nums_ok, "mảng nhảy")

phủ [a, b] bất kỳ và Jump Game II quy về video stitching: khớp vét cạn; 400 mảng nhảy


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi tập con | định nghĩa | `O(2^n·n·T)` | `O(T)` | oracle |
| 2. Tham lam `farthest` | Jump Game II | `O(n + T)` | `O(T)` | **đáp án chính** |
| 3. Sắp xếp + tham lam | gom theo `start ≤ reach` | `O(n log n)` | `O(1)` | không mảng phụ |
| 4. DP theo vị trí | `dp[t] = min dp[s] + 1` | `O(n·T)` | `O(T)` | dễ chứng minh |
| 5. BFS theo tầng | đồ thị trạng thái `reach` | `O(n·T)` | `O(T)` | trực quan |

Mọi cách khớp oracle định nghĩa trên `131 068` ca (mục 12D) và trên hàng nghìn ca ngẫu nhiên tới `n = time = 100`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [32]:
print(Solution().videoStitching([[0, 2], [4, 6], [8, 10], [1, 9], [1, 5], [5, 9]], 10), Solution().videoStitching([[0, 1], [1, 2]], 5),
      Solution().videoStitching([[0, 4]], 4), Solution().videoStitching([[0, 1], [1, 2]], 2), Solution().videoStitching([[1, 3]], 3))

3 -1 1 2 -1


### Bài 2 (tính tay)

`clips = [[0,3],[2,5],[4,8],[7,9]]`, `time = 9`. (a) Mảng `farthest`. (b) Mô phỏng tham lam từng bước. (c) Đáp án và các clip được chọn. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện tham lam bằng mảng `farthest`:

In [33]:
def farthest_todo(clips, time):
    farthest = [0] * (time + 1)
    for s, e in clips:
        pass            # TODO 1: nếu s <= time thì farthest[s] = max(farthest[s], e)
    count = cur_end = reach = 0
    for i in range(time):
        pass            # TODO 2: nếu i > reach trả -1; reach = max(reach, farthest[i])
        pass            # TODO 3: nếu i == cur_end: (nếu reach <= i trả -1) count += 1; cur_end = reach
    return count if reach >= time else -1


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(farthest_todo([x[:] for x in c], t) == w for (c, t), w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **quy hoạch động theo vị trí**:

In [34]:
class MySolution:
    def videoStitching(self, clips, time):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().videoStitching([x[:] for x in c], t) == w for (c, t), w in examples)

### Bài 5 (mini challenge): Jump Game II

Cho `nums` (`nums[i]` là bước nhảy tối đa từ `i`), trả về số lần nhảy tối thiểu từ `0` tới `n − 1` (luôn tới được). Gợi ý: quy về bài này bằng các "clip" `[i, i + nums[i]]` rồi phủ `[0, n − 1]`, hoặc dùng trực tiếp ba biến `count`, `cur_end`, `reach`.

In [35]:
def jump(nums):
    pass            # TODO


# Khi làm xong:
# assert jump([2, 3, 1, 1, 4]) == 2 and jump([2, 3, 0, 1, 4]) == 2 and jump([0]) == 0

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: ví dụ 1, 2 cho `3`, `-1`.
- Mức 2: `[[0,4]], 4` ⇒ `1`; `[[0,1],[1,2]], 2` ⇒ `2` (chạm đầu mút).
- Mức 3: `3 -1 1 2 -1`.

**Bài 2**

- Mức 1: `farthest[0] = 3`, `[2] = 5`, `[4] = 8`, `[7] = 9`.
- Mức 2: `reach` sau clip 1: `3`; các clip có `start ≤ 3`: `[0,3]`, `[2,5]` ⇒ `5`; rồi `start ≤ 5`: `[4,8]` ⇒ `8`; rồi `start ≤ 8`: `[7,9]` ⇒ `9`.
- Mức 3: đáp án `4` với các clip `[0,3]`, `[2,5]`, `[4,8]`, `[7,9]`.

**Bài 3**

- Mức 1: `farthest[s] = max(farthest[s], e)`.
- Mức 2: `if i > reach: return -1`; `reach = max(reach, farthest[i])`.
- Mức 3: `if i == cur_end: if reach <= i: return -1; count += 1; cur_end = reach`.

**Bài 4**

- Mức 1: `dp[0] = 0`, còn lại `∞`.
- Mức 2: `for t in 1..time: for (s, e): if s < t <= e: dp[t] = min(dp[t], dp[s] + 1)`.
- Mức 3: trả `dp[time]` hoặc `-1`.

**Bài 5**

- Mức 1: `cur_end` là biên của số bước hiện tại, `reach` là xa nhất tới được.
- Mức 2: với `i` từ `0` tới `n − 2`: `reach = max(reach, i + nums[i])`; nếu `i == cur_end` thì `jumps += 1`, `cur_end = reach`.
- Mức 3: trả `jumps`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [36]:
vals = (Solution().videoStitching([[0, 2], [4, 6], [8, 10], [1, 9], [1, 5], [5, 9]], 10), Solution().videoStitching([[0, 1], [1, 2]], 5),
        Solution().videoStitching([[0, 4]], 4), Solution().videoStitching([[0, 1], [1, 2]], 2), Solution().videoStitching([[1, 3]], 3))
print(vals)
assert vals == (3, -1, 1, 2, -1)
print("Bài 1 OK")

(3, -1, 1, 2, -1)
Bài 1 OK


### Bài 2

In [37]:
c, t = [[0, 3], [2, 5], [4, 8], [7, 9]], 9
farthest = [0] * (t + 1)
for s, e in c:
    farthest[s] = max(farthest[s], e)
print("farthest =", farthest)
assert farthest[0] == 3 and farthest[2] == 5 and farthest[4] == 8 and farthest[7] == 9
ans = Solution().videoStitching(c, t)
chosen = pick_clips(c, t)
print(ans, chosen)
assert ans == 4 == stitch_brute(c, t) and chosen == [[0, 3], [2, 5], [4, 8], [7, 9]] and covers(chosen, t)
print("Bài 2 OK")

farthest = [3, 0, 5, 0, 8, 0, 0, 9, 0, 0]
4 [[0, 3], [2, 5], [4, 8], [7, 9]]
Bài 2 OK


### Bài 3

In [38]:
def farthest_answer(clips, time):
    farthest = [0] * (time + 1)
    for s, e in clips:
        if s <= time:
            farthest[s] = max(farthest[s], e)
    count = cur_end = reach = 0
    for i in range(time):
        if i > reach:
            return -1
        reach = max(reach, farthest[i])
        if i == cur_end:
            if reach <= i:
                return -1
            count += 1
            cur_end = reach
    return count if reach >= time else -1


assert all(farthest_answer([x[:] for x in c], t) == w for (c, t), w in examples)
rng = random.Random(3)
for _ in range(500):
    T = rng.randint(1, 8)
    clips = random_clips(rng, rng.randint(1, 8), T, 4)
    assert farthest_answer(clips, T) == stitch_brute(clips, T)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [39]:
class MySolutionAnswer:
    def videoStitching(self, clips, time):
        INF = float("inf")
        dp = [INF] * (time + 1)
        dp[0] = 0
        for t in range(1, time + 1):
            for s, e in clips:
                if s < t <= e:
                    dp[t] = min(dp[t], dp[s] + 1)
        return dp[time] if dp[time] != INF else -1


assert all(MySolutionAnswer().videoStitching([x[:] for x in c], t) == w for (c, t), w in examples)
rng = random.Random(4)
for _ in range(500):
    T = rng.randint(1, 8)
    clips = random_clips(rng, rng.randint(1, 8), T, 4)
    assert MySolutionAnswer().videoStitching(clips, T) == stitch_brute(clips, T)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [40]:
def jump(nums):
    jumps = cur_end = reach = 0
    for i in range(len(nums) - 1):
        reach = max(reach, i + nums[i])
        if i == cur_end:
            jumps += 1
            cur_end = reach
    return jumps


assert jump([2, 3, 1, 1, 4]) == 2 and jump([2, 3, 0, 1, 4]) == 2 and jump([0]) == 0
rng = random.Random(5)
for _ in range(500):
    nums = [rng.randint(1, 4) for _ in range(rng.randint(1, 10))]
    assert jump(nums) == jump_brute(nums) == jump_game_ii(nums)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (ít đoạn nhất phủ kín một khoảng → tham lam "nhảy xa nhất"):**

```python
class Solution:
    def videoStitching(self, clips: List[List[int]], time: int) -> int:
        farthest = [0] * (time + 1)
        for s, e in clips:
            if s <= time:
                farthest[s] = max(farthest[s], e)
        count = cur_end = reach = 0
        for i in range(time):
            if i > reach:
                return -1
            reach = max(reach, farthest[i])
            if i == cur_end:
                if reach <= i:
                    return -1
                count += 1
                cur_end = reach
        return count if reach >= time else -1
```

**Checklist trước khi submit:**

- [ ] Nối clip khi `start ≤ reach` (đoạn đóng: chạm đầu mút vẫn nối)?
- [ ] `farthest[s] = max(end)` cho các clip cùng `start`; bỏ clip có `start > time`?
- [ ] Phát hiện khe hở (`i > reach`) và trường hợp không mở rộng được?
- [ ] Kiểm `reach ≥ time` cuối cùng, nếu không trả `-1`?
- [ ] `count` bắt đầu `0`; mỗi lần chốt clip `+1`?
- [ ] Đã thử ví dụ 1–3, `[[0,1],[1,2]]` với `time = 2`, `[[1,3]]`?
- [ ] Đúng tên `class Solution`, `videoStitching(self, clips, time)`, không in debug.